In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import statsmodels.stats.weightstats as smws

# Для воспроизводимости результатов зафиксируем seed
np.random.seed(42)

# Настройка графиков
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
print("=== ЗАДАЧА 1 ===")

# 1. Генерация выборки (n=50, mu=175, sigma=8)
n1 = 50
true_mu = 175
true_sigma = 8
sample_heights = np.random.normal(loc=true_mu, scale=true_sigma, size=n1)

# 2. Выборочное среднее и стандартное отклонение
sample_mean = np.mean(sample_heights)
sample_std = np.std(sample_heights, ddof=1) # ddof=1 для несмещенной оценки
print(f"Выборочное среднее: {sample_mean:.2f} см")
print(f"Выборочное станд. отклонение: {sample_std:.2f} см")

# 3. 95% ДИ при ИЗВЕСТНОМ стандартном отклонении (Z-интервал)
alpha = 0.05
z_critical = stats.norm.ppf(1 - alpha/2) # квантиль нормального распределения
margin_error_known = z_critical * (true_sigma / np.sqrt(n1))
ci_known_manual = (sample_mean - margin_error_known, sample_mean + margin_error_known)
print(f"\n3. ДИ для среднего (известное sigma), вручную: [{ci_known_manual[0]:.2f}, {ci_known_manual[1]:.2f}]")

# Проверка через библиотеку (scipy.stats.norm.interval)
ci_known_lib = stats.norm.interval(confidence=1-alpha, loc=sample_mean, scale=true_sigma/np.sqrt(n1))
print(f"   ДИ для среднего (известное sigma), библиотека: [{ci_known_lib[0]:.2f}, {ci_known_lib[1]:.2f}]")

# 4. 95% ДИ при НЕИЗВЕСТНОМ стандартном отклонении (t-интервал)
t_critical = stats.t.ppf(1 - alpha/2, df=n1-1)
margin_error_unknown = t_critical * (sample_std / np.sqrt(n1))
ci_unknown_manual = (sample_mean - margin_error_unknown, sample_mean + margin_error_unknown)
print(f"\n4. ДИ для среднего (неизвестное sigma), вручную: [{ci_unknown_manual[0]:.2f}, {ci_unknown_manual[1]:.2f}]")

# Проверка через библиотеку (scipy.stats.t.interval или statsmodels)
ci_unknown_lib = stats.t.interval(confidence=1-alpha, df=n1-1, loc=sample_mean, scale=sample_std/np.sqrt(n1))
# Альтернатива через statsmodels:
# ci_unknown_lib_sm = smws.DescrStatsW(sample_heights).tconfint_mean(alpha=alpha)
print(f"   ДИ для среднего (неизвестное sigma), библиотека: [{ci_unknown_lib[0]:.2f}, {ci_unknown_lib[1]:.2f}]")

# 5. Сравнение ширины интервалов
width_known = ci_known_manual[1] - ci_known_manual[0]
width_unknown = ci_unknown_manual[1] - ci_unknown_manual[0]
print(f"\n5. Ширина ДИ (известное sigma): {width_known:.2f}")
print(f"   Ширина ДИ (неизвестное sigma): {width_unknown:.2f}")
print(f"   Вывод: ДИ с неизвестной дисперсией шире, так как t-распределение имеет более тяжелые хвосты, чем нормальное, что учитывает дополнительную неопределенность оценки sigma.")

In [ ]:
print("\n=== ЗАДАЧА 2 ===")

# 1. Генерация данных
n2 = 30
temps = np.random.normal(loc=22, scale=3, size=n2)

# 1.b Добавление выбросов в 3 случайных дня
outlier_indices = np.random.choice(n2, size=3, replace=False)
# Случайно добавляем +5 или -5
temps[outlier_indices] += np.random.choice([-5, 5], size=3)

print(f"Исходные данные (с выбросами): {np.round(temps, 1)}")

# 2. 95% ДИ для средней температуры (исходные данные)
# Используем t-интервал, так как дисперсия неизвестна и есть выбросы
ci_temp_raw = stats.t.interval(0.95, df=n2-1, loc=np.mean(temps), scale=np.std(temps, ddof=1)/np.sqrt(n2))
print(f"\n2. ДИ для температуры (с выбросами): [{ci_temp_raw[0]:.2f}, {ci_temp_raw[1]:.2f}]")

# 3. Функция удаления выбросов по правилу 1.5 * IQR
def remove_outliers_iqr(data):
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return data[(data >= lower_bound) & (data <= upper_bound)]

cleaned_temps = remove_outliers_iqr(temps)
print(f"\n3. Данные после очистки (удалено {n2 - len(cleaned_temps)} выбросов): {np.round(cleaned_temps, 1)}")

# 4. ДИ для очищенных данных и сравнение
n_clean = len(cleaned_temps)
ci_temp_clean = stats.t.interval(0.95, df=n_clean-1, loc=np.mean(cleaned_temps), scale=np.std(cleaned_temps, ddof=1)/np.sqrt(n_clean))
print(f"\n4. ДИ для температуры (очищенные): [{ci_temp_clean[0]:.2f}, {ci_temp_clean[1]:.2f}]")
print(f"   Ширина исходного ДИ: {ci_temp_raw[1] - ci_temp_raw[0]:.2f}")
print(f"   Ширина очищенного ДИ: {ci_temp_clean[1] - ci_temp_clean[0]:.2f}")
print("   Вывод: После удаления выбросов доверительный интервал стал уже, а его центр сместился ближе к истинному среднему (22), так как выбросы искажали оценку среднего и увеличивали дисперсию.")

In [ ]:
print("\n=== ЗАДАЧА 3 ===")

# 1. Генерация выборки (параметры приняты как mu=50, sigma=10 из-за пропуска в тексте)
n3 = 60
true_var = 10**2 # 100
sample_var_data = np.random.normal(loc=50, scale=10, size=n3)

# 2. Выборочная дисперсия и станд. отклонение (несмещенные, ddof=1)
s2 = np.var(sample_var_data, ddof=1)
s = np.std(sample_var_data, ddof=1)
print(f"Выборочная дисперсия (s^2): {s2:.2f}")
print(f"Выборочное станд. отклонение (s): {s:.2f}")

# 3. 95% ДИ для дисперсии через хи-квадрат
df = n3 - 1
chi2_lower = stats.chi2.ppf(alpha/2, df)       # левый квантиль (меньшее значение)
chi2_upper = stats.chi2.ppf(1 - alpha/2, df)   # правый квантиль (большее значение)

# Формула: [(n-1)s^2 / chi2_upper, (n-1)s^2 / chi2_lower]
ci_var_lower = (df * s2) / chi2_upper
ci_var_upper = (df * s2) / chi2_lower
ci_var_manual = (ci_var_lower, ci_var_upper)
print(f"\n3. ДИ для дисперсии (вручную): [{ci_var_manual[0]:.2f}, {ci_var_manual[1]:.2f}]")

# 4. 95% ДИ для стандартного отклонения (корень из границ ДИ для дисперсии)
ci_std_manual = (np.sqrt(ci_var_lower), np.sqrt(ci_var_upper))
print(f"4. ДИ для станд. отклонения (вручную): [{ci_std_manual[0]:.2f}, {ci_std_manual[1]:.2f}]")

# 5. Сравнение ширины
print(f"\n5. Ширина ДИ для дисперсии: {ci_var_manual[1] - ci_var_manual[0]:.2f}")
print(f"   Ширина ДИ для станд. отклонения: {ci_std_manual[1] - ci_std_manual[0]:.2f}")
print("   (Сравнивать абсолютную ширину некорректно из-за разных единиц измерения, но относительная точность оценивается аналогично).")

# 6. Проверка через scipy.stats.bootstrap (альтернативный способ, упомянутый в задании)
# Примечание: bootstrap дает приближенный результат, chi2 - точный для нормального распределения
res_boot_var = stats.bootstrap((sample_var_data,), np.var, confidence_level=0.95, method='percentile')
print(f"\n6. Проверка ДИ для дисперсии через bootstrap (percentile): [{res_boot_var.confidence_interval.low:.2f}, {res_boot_var.confidence_interval.high:.2f}]")

In [ ]:
print("\n=== ЗАДАЧА 4 ===")

# 1. Генерация данных (Гамма-распределение)
n4 = 40
delivery_times = np.random.gamma(shape=2, scale=10, size=n4)

# 2. Визуализация
plt.hist(delivery_times, bins=15, edgecolor='black', alpha=0.7)
plt.title('Распределение времени доставки еды')
plt.xlabel('Время (минуты)')
plt.ylabel('Частота')
plt.axvline(np.median(delivery_times), color='red', linestyle='dashed', linewidth=2, label=f'Медиана = {np.median(delivery_times):.1f}')
plt.legend()
plt.show()

# 3. Базовый бутстреп ВРУЧНУЮ для медианы (90% ДИ)
n_boot = 10000
boot_medians = np.zeros(n_boot)

for i in range(n_boot):
    # Выборка с возвращением того же размера
    bootstrap_sample = np.random.choice(delivery_times, size=n4, replace=True)
    boot_medians[i] = np.median(bootstrap_sample)

# 90% ДИ соответствует 5-му и 95-му перцентилям
ci_boot_manual = np.percentile(boot_medians, [5, 95])
print(f"3. Базовый бутстреп ДИ для медианы (вручную): [{ci_boot_manual[0]:.2f}, {ci_boot_manual[1]:.2f}]")

# Проверка через scipy.stats.bootstrap (базовый метод 'percentile')
ci_boot_scipy = stats.bootstrap((delivery_times,), np.median, confidence_level=0.90, method='percentile', random_state=42)
print(f"   Базовый бутстреп ДИ для медианы (scipy): [{ci_boot_scipy.confidence_interval.low:.2f}, {ci_boot_scipy.confidence_interval.high:.2f}]")

In [ ]:
print("\n=== ЗАДАЧА 5 ===")

# 1. Используем данные из Задачи 4
theta_hat = np.median(delivery_times) # Оценка на исходной выборке

# Шаг 1. Коррекция смещения (bias correction, z0)
# Доля бутстреп-оценок, которые меньше исходной оценки
prop_less = np.mean(boot_medians < theta_hat)
# Избегаем деления на 0 или log(0), ограничивая вероятность
prop_less = np.clip(prop_less, 1e-9, 1 - 1e-9)
z0 = stats.norm.ppf(prop_less)
print(f"Шаг 1. Коэффициент смещения (z0): {z0:.4f}")

# Шаг 2. Коэффициент ускорения (acceleration, a) через jackknife
jackknife_estimates = np.zeros(n4)
for i in range(n4):
    # Выборка без i-го наблюдения
    jack_sample = np.delete(delivery_times, i)
    jackknife_estimates[i] = np.median(jack_sample)

jack_mean = np.mean(jackknife_estimates)
numerator = np.sum((jack_mean - jackknife_estimates)**3)
denominator = 6 * (np.sum((jack_mean - jackknife_estimates)**2))**1.5

# Защита от деления на ноль, если все jackknife оценки равны (редко, но бывает при медиане)
if denominator == 0:
    a = 0
else:
    a = numerator / denominator
print(f"Шаг 2. Коэффициент ускорения (a): {a:.4f}")

# Шаг 3. Скорректированные квантили для 90% ДИ (alpha = 0.10)
alpha = 0.10
z_alpha_2 = stats.norm.ppf(alpha / 2)       # ~ -1.645
z_1_alpha_2 = stats.norm.ppf(1 - alpha / 2) # ~ 1.645

# Формулы из методички:
# alpha_1 = Phi(z0 + (z0 + z_{alpha/2}) / (1 - a * (z0 + z_{alpha/2})))
# alpha_2 = Phi(z0 + (z0 + z_{1-alpha/2}) / (1 - a * (z0 + z_{1-alpha/2})))

def bca_quantile(z0, a, z_alpha):
    numerator = z0 + z_alpha
    denominator = 1 - a * numerator
    return stats.norm.cdf(z0 + (numerator / denominator))

alpha_1 = bca_quantile(z0, a, z_alpha_2)
alpha_2 = bca_quantile(z0, a, z_1_alpha_2)

print(f"Шаг 3. Скорректированные уровни квантилей: alpha_1 = {alpha_1:.4f}, alpha_2 = {alpha_2:.4f}")

# Шаг 4. Границы ДИ
ci_bca_manual = (
    np.percentile(boot_medians, 100 * alpha_1),
    np.percentile(boot_medians, 100 * alpha_2)
)
print(f"\nШаг 4. ДИ для медианы (BCa вручную): [{ci_bca_manual[0]:.2f}, {ci_bca_manual[1]:.2f}]")

# 3. Сравнение результатов
print("\n=== СРАВНЕНИЕ МЕТОДОВ ДЛЯ 90% ДИ МЕДИАНЫ ===")
print(f"1. Базовый (вручную)       : [{ci_boot_manual[0]:.2f}, {ci_boot_manual[1]:.2f}] (ширина: {ci_boot_manual[1]-ci_boot_manual[0]:.2f})")
print(f"2. Базовый (scipy)         : [{ci_boot_scipy.confidence_interval.low:.2f}, {ci_boot_scipy.confidence_interval.high:.2f}]")

# Готовый BCa из scipy
ci_bca_scipy = stats.bootstrap((delivery_times,), np.median, confidence_level=0.90, method='BCa', random_state=42)
print(f"3. Ускоренный BCa (scipy)  : [{ci_bca_scipy.confidence_interval.low:.2f}, {ci_bca_scipy.confidence_interval.high:.2f}] (ширина: {ci_bca_scipy.confidence_interval.high - ci_bca_scipy.confidence_interval.low:.2f})")
print(f"4. Ускоренный BCa (вручную): [{ci_bca_manual[0]:.2f}, {ci_bca_manual[1]:.2f}]")

print("\nВывод: Метод BCa корректирует границы интервала, учитывая асимметрию распределения бутстреп-оценок (через z0) и нелинейность статистики (через a). Для гамма-распределения (которое асимметрично) BCa часто дает более точный и немного смещенный относительно центра интервал по сравнению с базовым перцентильным методом.")